# NYUAD Academic Landscape
### Aashma Varma

Human-Centered Data Science – Mini Project

Professor Minsu Park

This notebook maps the research landscape of NYUAD faculty by converting their research descriptions into vectors, clustering similar researchers together, and visualizing them in an interactive 2D plot.

Core Idea: Faculty who share research interests will appear closer together on the plot.

## 1. Import Libraries

In [ ]:
import pandas as pd                                    # for working with tabular data (our faculty dataset)
import textwrap                                        # for wrapping long text in hover boxes
from sentence_transformers import SentenceTransformer  # converts text into vectors (embeddings)
from sklearn.cluster import KMeans                     # groups similar vectors into clusters
from sklearn.preprocessing import normalize            # scales vectors to unit length for fair comparison
import umap                                            # reduces high-dimensional vectors to 2D for plotting
import plotly.express as px                            # interactive plotting library
import numpy as np

## 2. Load and Clean Data

We load the faculty dataset scraped from NYUAD's faculty profile pages. Each row represents one faculty member, with columns for their name, job title, division, and research description.

A few cleaning steps are needed before we can use the data:
- The division labels came out of the scraper with hyphens (e.g. `social-science`) — we convert them to title case (`Social Science`)
- Some faculty have no research description at all — we filter them out since there's nothing to embed
- We wrap long text fields so hover boxes in the final plot stay a consistent width

In [ ]:
file = pd.read_csv("faculty_data.csv")

print("Columns found:", file.columns.tolist())
print(f"Total faculty loaded: {len(file)}")

In [ ]:
# Clean up division labels: 'arts-and-humanities' -> 'Arts And Humanities'
file["Faculty"] = (
    file["Faculty"].str.replace("-", " ").str.title()
)

print("Faculty divisions found:")
print(file["Faculty"].value_counts())

In [ ]:
# Strip whitespace from research text
file["Research Text"] = (
    file["Full Research Paragraph"].fillna("").str.strip()
)

# Remove faculty with no research description.
# Keeping them in would mean embedding an empty string, which produces a
# near-zero vector — these faculty would cluster together for the wrong
# reason (missing data, not similar research).
file = file[file["Research Text"] != ""].reset_index(drop=True)

print(f"{len(file)} faculty members have research descriptions.")

In [ ]:
# Wrap long text so hover boxes in the plot stay readable
file["Research Text"] = file["Research Text"].apply(
    lambda t: "<br>".join(textwrap.wrap(t, width=60))
)

file["Job Title"] = file["Job Title"].apply(
    lambda t: "<br>".join(textwrap.wrap(t, width=60))
)

## 3. Sentence Embeddings

[Sentence embeddings info](https://osanseviero.github.io/hackerllama/blog/posts/sentence_embeddings/)

To compare research descriptions mathematically, we need to convert them from text into numbers. We use a technique called **sentence embeddings** — a pre-trained neural network reads each paragraph and outputs a vector of 384 numbers that captures the *meaning* of the text.

**Why embeddings instead of simpler approaches like TF-IDF?**  
TF-IDF compares documents by counting word overlap. But two researchers could study very similar things using completely different vocabulary — for example, one might write about *"neural circuits"* and another about *"brain networks"*. TF-IDF would treat these as unrelated. An embedding model, having been trained on large amounts of text, understands that these phrases mean similar things and places them close together in vector space.

**Model choice:** `all-MiniLM-L6-v2` from the `sentence-transformers` library. It's popular and lightweight.

**Why normalize?**  
After encoding, vectors have different magnitudes based on the length of text, but we only care about their direction which represents meaning. Normalizing scales every vector to length 1, so we can measure the cosine similarity of the vectors (the angle between two vectors).

*Cosine similarity ranges from -1 to 1, where 1 means vectors point in identical directions (maximum similarity), 0 means they're unrelated, and -1 means they're opposite. Without normalization, longer text would get artificially higher scores just from having more words. Normalization ensures every comparison focuses purely on meaning (direction) and not paragraph length (magnitude). This ensures fair comparison across all faculty regardless of how much text they wrote.*

[Cosine similarity info 1](https://www.geeksforgeeks.org/dbms/cosine-similarity/) [Cosine similarity info 2](https://www.ibm.com/think/topics/cosine-similarity)

In [ ]:
print("Loading embedding model...")
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Generating embeddings...")
embeddings = model.encode(
    file["Full Research Paragraph"].tolist(),
    show_progress_bar=True,
    # process 32 texts at a time to manage memory
    batch_size=32
)

# Normalize so all vectors have magnitude 1 (enables cosine similarity)
embeddings = normalize(embeddings)

# embeddings.shape = (387, 384)
# (no. of faculty members, no. of values in embeddings)
print(f"\nEmbeddings shape: {embeddings.shape}")
print(f"Each faculty is now represented by a vector of {embeddings.shape[1]} numbers.")

## 4. K-Means Clustering

[ML algo info](https://pythongeeks.org/machine-learning-algorithms/)
[K-Means info](https://www.geeksforgeeks.org/machine-learning/k-means-clustering-introduction/)

Now that each faculty member is a vector, we use **K-Means** to group similar researchers together.

K-Means works by:
1. Randomly placing K "centroids" (cluster centers) in vector space
2. Assigning each faculty to their nearest centroid
3. After assignment, the centroid of each cluster is recalculated by averaging the points within it
4. Process repeated until centroids no longer change or max no. of iterations reached

**Why K=8?**  
This was chosen manually based on the number of broad research areas we expected to see (e.g. sciences, humanities, social sciences, engineering, etc.). A more principled approach would be the **elbow method** — running K-Means for different values of K, plotting the "inertia" (how spread out clusters are), and picking the K where improvement starts to flatten out.

**`n_init=10`** means K-Means runs 10 times with different random starting centroids and picks the best result. Some random starts will lead to better solutions than others, so by running 10 times at random starting points, we can pick the best quality attempt.

[N-init info](https://www.pythonalchemist.com/blog/kmeans-centroid-trap)

In [ ]:
K = 8
print(f"Running K-Means with K={K} clusters...")

kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
file["Cluster"] = kmeans.fit_predict(embeddings).astype(str)

print("\nFaculty per cluster:")
print(file["Cluster"].value_counts().sort_index())

## 5. UMAP — Compressing to 2D

Our embeddings are 384-dimensional, so they are impossible to visualize directly. We use **UMAP** to compress them into 2D coordinates while preserving the neighborhood structure: faculty who were close together in 384D should still be close together in 2D.

[Info in layman terms...](https://www.reddit.com/r/datascience/comments/wy1rmk/pca_vs_umap_vs_tsne_on_a_very_layman_level_what/)

**Why UMAP instead of PCA or t-SNE?**  
- **PCA** is fast and simple but only captures linear relationships, it often flattens out interesting structure
- **t-SNE** is good at preserving local structure but is slow and doesn't scale well to large datasets
- **UMAP** is fast, scales well, and preserves both local *and* global structure better than t-SNE

**Key parameters:**
- `n_neighbors=8` — how many nearby points each point considers when learning the layout. Smaller values focus on local structure (tighter clusters); larger values give a more global view
- `min_dist=0.6` — how spread out the points are in 2D. Higher values create a more spread-out, readable map; lower values pack clusters more tightly

In [ ]:
print("Running UMAP (compressing 384 dimensions → 2D)...")

reducer = umap.UMAP(
    n_neighbors=8,
    min_dist=0.6,
    random_state=42
)

coords_2d = reducer.fit_transform(embeddings)

file["x"] = coords_2d[:, 0]
file["y"] = coords_2d[:, 1]

print(f"Done. Each faculty now has an (x, y) coordinate for plotting.")

## 6. Interactive Scatter Plot

Finally, we plot each faculty member as a dot on a 2D map. Points are coloured by their K-Means cluster. Hovering over a dot shows their name, job title, division, and research description.

Faculty who appear close together have research descriptions that are semantically similar, even if they're in different divisions.

In [ ]:
print("Building interactive plot...")

figure = px.scatter(
    file,
    x="x",
    y="y",
    color="Cluster",
    hover_name="Name",
    hover_data={
        "Job Title": True,
        "Faculty": True,
        "Research Text": True,
        "x": False,
        "y": False,
        "Cluster": False
    },
    title="NYUAD Academic Landscape",
    labels={"Cluster": "Research Cluster", "Faculty": "Faculty Division"},
    width=1100,
    height=750,
    template="plotly_white"
)

figure.update_traces(
    marker=dict(size=7, line=dict(width=0.8, color="white")),
    selector=dict(mode="markers")
)

figure.update_layout(
    showlegend=False,
    font_family="Arial",
    title_font_size=18,
    xaxis=dict(showticklabels=False, showgrid=False, zeroline=False, title=""),
    yaxis=dict(showticklabels=False, showgrid=False, zeroline=False, title=""),
    plot_bgcolor="#fafafa",
)

# Save as a standalone HTML file anyone can open in a browser
figure.write_html("nyuad_landscape.html")
print("Saved to nyuad_landscape.html")

figure.show()